In [1]:
# Install COLMAP
!apt-get update
!apt-get install -y colmap

Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 https://cli.github.com/packages stable/main amd64 Packages [354 B]
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:9 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,701 kB]
Get:10 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Hit:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:12 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,959 kB]
Hit:13 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ub

In [2]:
# 1. Mounting Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. COLMAP and Building Tools
!apt-get update
!apt-get install -y colmap libglue-dev libsuitesparse-dev

%cd /content
!git clone --recursive https://github.com/camenduru/gaussian-splatting
%cd /content/gaussian-splatting
!pip install plyfile tqdm open3d

Mounted at /content/drive
Hit:1 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:4 https://cli.github.com/packages stable InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry

In [1]:
%cd /content/gaussian-splatting/submodules/simple-knn
!sed -i '1i #include <cfloat>' simple_knn.cu
!python setup.py install

/content/gaussian-splatting/submodules/simple-knn
running install
/usr/local/lib/python3.12/dist-packages/setuptools/_distutils/cmd.py:66: SetuptoolsDeprecationWarning: setup.py install is deprecated.
!!

        ********************************************************************************
        Please avoid running ``setup.py`` directly.
        Instead, use pypa/build, pypa/installer or other
        standards-based tools.

        See https://blog.ganssle.io/articles/2021/10/setup-py-deprecated.html for details.
        ********************************************************************************

!!
  self.initialize_options()
/usr/local/lib/python3.12/dist-packages/setuptools/_distutils/cmd.py:66: EasyInstallDeprecationWarning: easy_install command is deprecated.
!!

        ********************************************************************************
        Please avoid running ``setup.py`` and ``easy_install``.
        Instead, use pypa/build, pypa/installer or other

In [2]:
%cd /content/gaussian-splatting/submodules/diff-gaussian-rasterization
!python setup.py install

%cd /content/gaussian-splatting

/content/gaussian-splatting/submodules/diff-gaussian-rasterization
running install
/usr/local/lib/python3.12/dist-packages/setuptools/_distutils/cmd.py:66: SetuptoolsDeprecationWarning: setup.py install is deprecated.
!!

        ********************************************************************************
        Please avoid running ``setup.py`` directly.
        Instead, use pypa/build, pypa/installer or other
        standards-based tools.

        See https://blog.ganssle.io/articles/2021/10/setup-py-deprecated.html for details.
        ********************************************************************************

!!
  self.initialize_options()
/usr/local/lib/python3.12/dist-packages/setuptools/_distutils/cmd.py:66: EasyInstallDeprecationWarning: easy_install command is deprecated.
!!

        ********************************************************************************
        Please avoid running ``setup.py`` and ``easy_install``.
        Instead, use pypa/build, pypa/i

In [3]:
import os
from PIL import Image

#
DRIVE_IMAGES = "/content/drive/MyDrive/Vision/Dori/images"
DRIVE_SPARSE = "/content/drive/MyDrive/Vision/Dori/sparse/0"
LOCAL_DATA = "/content/ready_for_dl"

# 1. Resizing Images
os.makedirs("/content/images_fixed", exist_ok=True)
print("Resizing images to 848x478...")
for f in os.listdir(DRIVE_IMAGES):
    if f.lower().endswith(('.jpg', '.jpeg', '.png')):
        img = Image.open(os.path.join(DRIVE_IMAGES, f))
        img = img.resize((848, 478), Image.LANCZOS)
        img.save(os.path.join("/content/images_fixed", f))

#
!rm -rf {LOCAL_DATA}
os.makedirs(LOCAL_DATA, exist_ok=True)
print("Running COLMAP Undistorter...")
!colmap image_undistorter \
    --image_path /content/images_fixed \
    --input_path {DRIVE_SPARSE} \
    --output_path {LOCAL_DATA} \
    --output_type COLMAP

#
#
!mkdir -p {LOCAL_DATA}/sparse/0
!mv {LOCAL_DATA}/sparse/*.bin {LOCAL_DATA}/sparse/0/ 2>/dev/null

# Final Verification
if os.path.exists(f"{LOCAL_DATA}/images") and len(os.listdir(f"{LOCAL_DATA}/images")) > 0:
    print(f" Success! {len(os.listdir(f'{LOCAL_DATA}/images'))} images are ready.")
    print("Data ready for Hybrid Training.")
else:
    print(" ERROR: Undistorter failed or images were not generated. Check COLMAP output above.")

Resizing images to 848x478...
Running COLMAP Undistorter...

Reading reconstruction

 => Reconstruction with 128 images and 15423 points

Image undistortion

Undistorting image [1/128]
Undistorting image [2/128]
Undistorting image [3/128]
Undistorting image [4/128]
Undistorting image [5/128]
Undistorting image [6/128]
Undistorting image [7/128]
Undistorting image [8/128]
Undistorting image [9/128]
Undistorting image [10/128]
Undistorting image [11/128]
Undistorting image [12/128]
Undistorting image [13/128]
Undistorting image [14/128]
Undistorting image [15/128]
Undistorting image [16/128]
Undistorting image [17/128]
Undistorting image [18/128]
Undistorting image [19/128]
Undistorting image [20/128]
Undistorting image [21/128]
Undistorting image [22/128]
Undistorting image [23/128]
Undistorting image [24/128]
Undistorting image [25/128]
Undistorting image [26/128]
Undistorting image [27/128]
Undistorting image [28/128]
Undistorting image [29/128]
Undistorting image [30/128]
Undistortin

In [4]:
# dependencies
!pip install plyfile tqdm

# simple-knn (with the patch for Python 3.12)
%cd /content/gaussian-splatting/submodules/simple-knn
!sed -i '1i #include <cfloat>' simple_knn.cu
!python setup.py install

# Build the Rasterizer
%cd /content/gaussian-splatting/submodules/diff-gaussian-rasterization
!python setup.py install

DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/diff_gaussian_rasterization-0.0.0-py3.12-linux-x86_64.egg is deprecated. pip 24.3 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/simple_knn-0.0.0-py3.12-linux-x86_64.egg is deprecated. pip 24.3 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
/content/gaussian-splatting/submodules/simple-knn
running install
/usr/local/lib/python3.12/dist-packages/setuptools/_distutils/cmd.py:66: SetuptoolsDeprecationWarning: setup.py install is deprecated.
!!

        ********************************************************************************
        Please avoid running ``setup.py`` directly.
        Instead, use pypa/build, pypa

In [10]:
%cd /content/gaussian-splatting

# Training for 7000 iterations
# -s points to the folder we just prepared
!python train.py -s /content/ready_for_dl --iterations 30000

/content/gaussian-splatting
2026-06-01 15:51:22.598259: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Optimizing 
Output folder: ./output/a09d1b41-5 [01/06 15:51:27]
Reading camera 128/128 [01/06 15:51:28]
Loading Training Cameras [01/06 15:51:28]
Loading Test Cameras [01/06 15:51:30]
Number of points at initialisation :  15423 [01/06 15:51:30]
Training progress:  23% 7000/30000 [06:08<27:09, 14.11it/s, Loss=0.0295001]
[ITER 7000] Evaluating train: L1 0.017685669474303722 PSNR 32.21006813049316 [01/06 15:57:42]

[ITER 7000] Saving Gaussians [01/06 15:57:42]
Training progress: 100% 30000/30000 [35:28<00:00, 14.10it/s, Loss=0.0228165]

[ITER 30000] Evaluating train: L1 0.014008131437003614 PSNR 34.20622940063477 [01/06 16:27:01]

[ITER 30000] Sav

In [12]:
!cp -r /content/gaussian-splatting/output \
"/content/drive/MyDrive/GaussianOutput/7k"